# SolarSense EMS — Industrial Load & LESCO MDI Spike Analysis
### Phase 2 Milestone 5: Exploratory Data Analysis & Baseline Loss Reconciliation
**Location:** Lahore Industrial Area, Punjab, Pakistan
**Facility:** Textile Spinning & Weaving Mill (500 kW LESCO B3 Sanctioned Ceiling)
**Period:** July 1, 2025 – October 7, 2026 (464 Days / 11,136 Hourly Observations)

---

## 1. Executive Summary & Problem Formulation
Under the **NEPRA / LESCO B3 Industrial Tariff**, high-voltage consumers face three major cost drivers:
1. **Time-of-Use (ToU) Energy Rates:** 38.00 PKR/kWh off-peak vs. 60.00 PKR/kWh during the 4-hour seasonal peak window.
2. **Maximum Demand Indicator (MDI) Ratchet:** Exceeding the 500 kW sanctioned limit incurs a fine of **1,200.00 PKR per peak kW**, billed on the highest instantaneous peak recorded that calendar month.
3. **Load Shedding / Blackout Diesel Generation:** During feeder outages, unmet factory demand must be served by emergency diesel generators at **95.00 PKR/kWh**.

### The Critical Physics Law: Outage MDI Immunity
Because the LESCO feeder line is de-energized during blackouts (`outage_flag == 1`), the factory meter registers `0.0 kW` grid draw, so the factory cannot incur an MDI penalty when the grid is dead. This notebook verifies that invariant and shows that the historical **2.36 Million PKR** baseline (`demand_penalty_pkr`, asserted in `tests/test_feature_store.py`) comes **entirely from grid-connected hours**, with none from blackouts.

> The factory load (Dataset C) is **synthetic** (`src/data_engineering/build_factory_load.py`, seed 42), so the MDI spikes are driven by the synthetic load (see `GUIDE.md` §11).


In [ ]:
import sys
from decimal import ROUND_HALF_EVEN, Decimal
from pathlib import Path

# Add project root to path (works when launched from notebooks/ or the repo root)
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from src.data_engineering.feature_store import EMSDataRepository, TariffPolicy

# Explicit paths: EMSDataRepository defaults are relative to the repo root.
repo = EMSDataRepository(
    csv_path=PROJECT_ROOT / "data" / "processed" / "master_ems_timeseries.csv",
    parquet_path=PROJECT_ROOT / "data" / "processed" / "master_ems_timeseries.parquet",
)
policy = TariffPolicy()
df = repo._df

SANCTIONED_KW = policy.sanctioned_load_kw
PENALTY_PKR_PER_KW = float(policy.mdi_penalty_per_kw_pkr)
PAISA = Decimal("0.01")
COLORS = {"night": "#3F51B5", "day": "#FF9800", "evening": "#4CAF50", "load": "#2196F3", "spike": "#F44336"}


def pkr_total(frame: pl.DataFrame, column: str) -> Decimal:
    """Exact PKR sum: vectorised integer-paisa addition, then Decimal (no float drift)."""
    paisa = frame.select((pl.col(column) * 100).round().cast(pl.Int64).sum()).item()
    return (Decimal(int(paisa or 0)) * PAISA).quantize(PAISA, rounding=ROUND_HALF_EVEN)


print("Canonical EMS Dataset Loaded:")
print(f"Total Rows: {df.height:,}")
print(f"Date Range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Total Outage Hours: {df['outage_flag'].sum()}")

## 2. Factory Load Distribution & Shift Analytics
The facility runs two production shifts (the schedule used to synthesize the load in `src/data_engineering/build_factory_load.py`):
- **Day Shift (Peak Production):** 08:00 – 17:00
- **Evening Shift (Finishing / Packaging):** 17:00 – 22:00
- **Off-Shift (Base Load):** 22:00 – 08:00, with machinery ramping up from 06:00

Fridays run at 65% and Sundays at 40% of the weekday shift load.


In [ ]:
# Add shift classification
df_shifts = df.with_columns([
    pl.col("timestamp").dt.hour().alias("hour"),
    pl.when(pl.col("timestamp").dt.hour().is_between(8, 16))
    .then(pl.lit("Day Shift (08-17h)"))
    .when(pl.col("timestamp").dt.hour().is_between(17, 21))
    .then(pl.lit("Evening Shift (17-22h)"))
    .otherwise(pl.lit("Off-Shift (22-08h)"))
    .alias("shift"),
    # weekday: 0=Mon ... 4=Fri, 6=Sun (Friday and Sunday run reduced shifts)
    pl.when(pl.col("weekday") == 4).then(pl.lit("Friday"))
    .when(pl.col("weekday") == 6).then(pl.lit("Sunday"))
    .otherwise(pl.lit("Working day"))
    .alias("day_type"),
])

shift_stats = df_shifts.group_by("shift").agg([
    pl.col("factory_load_kw").mean().round(2).alias("mean_load_kw"),
    pl.col("factory_load_kw").median().round(2).alias("median_load_kw"),
    pl.col("factory_load_kw").max().round(2).alias("peak_load_kw"),
    pl.col("factory_load_kw").quantile(0.95).round(2).alias("p95_load_kw"),
]).sort("shift")

print("Factory Load Statistics by Operational Shift:")
print(shift_stats)

In [ ]:
DAY_TYPES = ["Working day", "Friday", "Sunday"]
SHIFTS = ["Day Shift (08-17h)", "Evening Shift (17-22h)", "Off-Shift (22-08h)"]

shift_by_day = (
    df_shifts.group_by(["shift", "day_type"])
    .agg(pl.col("factory_load_kw").mean())
    .pivot(on="day_type", index="shift", values="factory_load_kw")
    .sort("shift")
)
heat = (
    df_shifts.group_by(["hour", "month"])
    .agg(pl.col("factory_load_kw").mean())
    .pivot(on="month", index="hour", values="factory_load_kw")
    .sort("hour")
)
month_cols = sorted((c for c in heat.columns if c != "hour"), key=int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(SHIFTS))
for i, (day_type, color) in enumerate(zip(DAY_TYPES, ["#2196F3", "#FF9800", "#9E9E9E"])):
    axes[0].bar(x + (i - 1) * 0.27, shift_by_day[day_type].to_numpy(), width=0.27, color=color, label=day_type)
axes[0].set_xticks(x, SHIFTS)
axes[0].set_ylabel("Mean load (kW)")
axes[0].set_title("Mean load per shift by day type")
axes[0].legend()
axes[0].grid(alpha=0.3, axis="y")

im = axes[1].imshow(heat.select(month_cols).to_numpy(), aspect="auto", cmap="YlOrRd", origin="lower")
axes[1].set_xticks(range(len(month_cols)), month_cols)
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Hour of day")
axes[1].set_title("Mean load (kW): hour × month")
fig.colorbar(im, ax=axes[1], label="kW")
plt.tight_layout()
plt.show()

## 3. Historical MDI Spike Analysis (> 500 kW Ceiling)
We identify all hours where net grid import breached the sanctioned 500 kW ceiling during **grid-connected operation**.


In [ ]:
mdi_spikes = repo.get_mdi_spikes(threshold_kw=SANCTIONED_KW)
spike_penalty_pkr = pkr_total(mdi_spikes, "instantaneous_penalty_pkr")
print(f"Total Grid-Connected MDI Breaches: {mdi_spikes.height}")
print(f"Hourly MDI Penalty (sum of breaches): {spike_penalty_pkr:,} PKR")
print("Top 10 Worst Historical MDI Violations:")
print(mdi_spikes.head(10))

In [ ]:
spikes_by_shift = (
    mdi_spikes.join(df_shifts.select(["timestamp", "shift"]), on="timestamp")
    .group_by("shift").len().sort("shift")
)
print("MDI breaches by shift:")
print(spikes_by_shift)

monthly_penalty = (
    mdi_spikes.group_by(pl.col("timestamp").dt.strftime("%Y-%m").alias("month"))
    .agg(pl.col("instantaneous_penalty_pkr").sum().alias("penalty_pkr"))
    .sort("month")
)
spikes_by_hour = (
    pl.DataFrame({"hour": list(range(24))}, schema={"hour": mdi_spikes["hour"].dtype})
    .join(mdi_spikes.group_by("hour").len(), on="hour", how="left")
    .fill_null(0)
    .sort("hour")
)
worst_days = (
    mdi_spikes.group_by("date")
    .agg([
        pl.len().alias("spike_hours"),
        pl.col("grid_import_kw").max().round(2).alias("peak_import_kw"),
        pl.col("instantaneous_penalty_pkr").sum().round(2).alias("penalty_pkr"),
    ])
    .sort("penalty_pkr", descending=True)
    .head(10)
)


def plot_spike_timeseries(ax):
    ax.plot(df["timestamp"].dt.replace_time_zone(None).to_numpy(), df["grid_import_kw"].to_numpy(),
            color=COLORS["load"], lw=0.4, alpha=0.7, label="Grid import")
    ax.scatter(mdi_spikes["timestamp"].dt.replace_time_zone(None).to_numpy(), mdi_spikes["grid_import_kw"].to_numpy(),
               color=COLORS["spike"], s=12, zorder=3, label=f"MDI breach ({mdi_spikes.height} h)")
    ax.axhline(SANCTIONED_KW, color="red", ls="--", lw=1.5, label=f"Sanctioned limit ({SANCTIONED_KW:.0f} kW)")
    ax.set_ylabel("Grid import (kW)")
    ax.set_title(f"Grid import and MDI breaches — hourly penalty {spike_penalty_pkr:,.0f} PKR")
    ax.legend(loc="lower left", fontsize=8)
    ax.grid(alpha=0.3)


def plot_monthly_penalty(ax):
    ax.bar(monthly_penalty["month"].to_list(), monthly_penalty["penalty_pkr"].to_numpy() / 1e3, color=COLORS["spike"])
    ax.set_ylabel("Penalty (k PKR)")
    ax.set_title("MDI penalty by month")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(alpha=0.3, axis="y")


def plot_spikes_by_hour(ax):
    hours = spikes_by_hour["hour"].to_numpy()
    colors = np.where(
        (hours >= 8) & (hours < 17), COLORS["day"],
        np.where((hours >= 17) & (hours < 22), COLORS["evening"], COLORS["night"]),
    )
    ax.bar(hours, spikes_by_hour["len"].to_numpy(), color=colors)
    ax.set_xticks(range(0, 24, 2))
    ax.set_xlabel("Hour of day (Asia/Karachi PKT)")
    ax.set_ylabel("Breach hours")
    ax.set_title("MDI breaches by hour (orange = Day Shift, green = Evening Shift)")
    ax.grid(alpha=0.3, axis="y")


fig, ax = plt.subplots(figsize=(14, 5))
plot_spike_timeseries(ax)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
plot_monthly_penalty(axes[0])
plot_spikes_by_hour(axes[1])
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.barh(worst_days["date"].to_list()[::-1], worst_days["penalty_pkr"].to_numpy()[::-1] / 1e3, color=COLORS["spike"])
ax.set_xlabel("Penalty (k PKR)")
ax.set_title("Top 10 worst MDI penalty days")
ax.grid(alpha=0.3, axis="x")
plt.show()
worst_days

### Peak-shaving requirement (shave to 495 kW)
For each breach day: the battery energy (kWh) needed to hold grid import at **495 kW**, leaving a 5 kW safety margin under the 500 kW limit. With 1-hour steps, kW above the target equals kWh discharged. The worst day sets the minimum usable battery capacity, before inverter-power and depth-of-discharge limits.


In [ ]:
SHAVE_TARGET_KW = SANCTIONED_KW - 5.0
daily_shave = (
    df.filter(pl.col("outage_flag") == 0)
    .with_columns((pl.col("grid_import_kw") - SHAVE_TARGET_KW).clip(lower_bound=0).alias("shave_kwh"))
    .group_by("date")
    .agg([pl.col("shave_kwh").sum(), pl.col("shave_kwh").max().alias("shave_kw")])
    .filter(pl.col("shave_kwh") > 0)
)
shave_kwh = daily_shave["shave_kwh"]

print(f"Days needing shaving:       {daily_shave.height}")
print(f"Total shaving energy:       {shave_kwh.sum():,.1f} kWh")
print(f"Median / p95 / max per day: {shave_kwh.median():.1f} / {shave_kwh.quantile(0.95):.1f} / {shave_kwh.max():.1f} kWh")
print(f"Peak discharge power:       {daily_shave['shave_kw'].max():.1f} kW")

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.hist(shave_kwh.to_numpy(), bins=30, color=COLORS["load"], edgecolor="white")
ax.axvline(shave_kwh.quantile(0.95), color=COLORS["day"], ls="--", label=f"p95 = {shave_kwh.quantile(0.95):.0f} kWh")
ax.axvline(shave_kwh.max(), color="red", ls="--", label=f"max = {shave_kwh.max():.0f} kWh")
ax.set_xlabel("Battery energy needed per day (kWh)")
ax.set_ylabel("Days")
ax.set_title(f"Daily peak-shaving requirement to hold grid import at {SHAVE_TARGET_KW:.0f} kW")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 4. Reconciling the 2.36 Million PKR Baseline
Two checks against the pipeline's `demand_penalty_pkr` column:
1. **Outage MDI immunity:** blackout hours must contribute zero penalty, even where gross factory load exceeds 500 kW.
2. **Reproduction:** the grid-connected breaches from `get_mdi_spikes()` must add up to the full 2.36M PKR baseline.


In [ ]:
# Naive gross demand exceedance during blackouts
blackout_exceedances = df.filter(
    (pl.col("outage_flag") == 1) & (pl.col("factory_load_kw") > SANCTIONED_KW)
)

fake_penalties = float(
    ((blackout_exceedances["factory_load_kw"] - SANCTIONED_KW) * PENALTY_PKR_PER_KW).sum()
)
baseline_penalty_pkr = pkr_total(df, "demand_penalty_pkr")
blackout_penalty_pkr = pkr_total(df.filter(pl.col("outage_flag") == 1), "demand_penalty_pkr")

print(f"Blackout Hours with Gross Load > 500 kW: {blackout_exceedances.height}")
print(f"Phantom MDI Fines Avoided by Physics Invariant: {fake_penalties:,.2f} PKR")
print(f"Pipeline Penalty Charged During Blackouts: {blackout_penalty_pkr:,} PKR")
print(f"Actual Grid MDI Violations Counted by LESCO: {mdi_spikes.height}")
print(f"Pipeline Baseline (demand_penalty_pkr): {baseline_penalty_pkr:,} PKR")
print(f"Reproduced from Grid-Connected Breaches: {spike_penalty_pkr:,} PKR")

assert blackout_penalty_pkr == Decimal("0.00"), "MDI penalty charged during a blackout"
assert mdi_spikes.height == 169, f"expected 169 breaches, got {mdi_spikes.height}"
assert abs(spike_penalty_pkr - Decimal("2360353.20")) <= Decimal("1.0"), spike_penalty_pkr
assert abs(spike_penalty_pkr - baseline_penalty_pkr) <= Decimal("1.0"), (spike_penalty_pkr, baseline_penalty_pkr)

### Hourly sum vs. monthly ratchet
The 2.36M PKR baseline charges **every** breach hour. LESCO actually bills the MDI fine **once per calendar month, on that month's single highest peak** above 500 kW (the ratchet in `TariffPolicy.calculate_marginal_mdi_penalty`). On the same data the billed figure is much smaller. The optimizer's savings should be quoted against the method the bill actually uses.


In [ ]:
ratchet = (
    df.filter(pl.col("outage_flag") == 0)
    .group_by(pl.col("timestamp").dt.strftime("%Y-%m").alias("month"))
    .agg(pl.col("grid_import_kw").max().round(2).alias("peak_import_kw"))
    .filter(pl.col("peak_import_kw") > SANCTIONED_KW)
    .with_columns(((pl.col("peak_import_kw") - SANCTIONED_KW) * PENALTY_PKR_PER_KW).round(2).alias("ratchet_penalty_pkr"))
    .join(monthly_penalty.rename({"penalty_pkr": "hourly_sum_penalty_pkr"}), on="month", how="left")
    .sort("month")
)
ratchet_total_pkr = pkr_total(ratchet, "ratchet_penalty_pkr")

print(f"Hourly-sum method:      {spike_penalty_pkr:,} PKR")
print(f"Monthly-ratchet method: {ratchet_total_pkr:,} PKR")
ratchet

## 5. Visualizing Diurnal Load Curve & MDI Breaches
We plot the 24-hour diurnal factory load, solar generation, and the 500 kW sanctioned boundary. The shaded bands mark the three shifts. The export combines this curve with the breach timeline, monthly penalty and breach-hour charts.


In [ ]:
hourly_profiles = df.group_by(df["timestamp"].dt.hour().alias("hour")).agg([
    pl.col("factory_load_kw").mean().alias("avg_load_kw"),
    pl.col("factory_load_kw").quantile(0.95).alias("p95_load_kw"),
    pl.col("factory_load_kw").max().alias("max_load_kw"),
    pl.col("effective_solar_kw").mean().alias("avg_solar_kw"),
]).sort("hour")

hours = hourly_profiles["hour"].to_numpy()
avg_load = hourly_profiles["avg_load_kw"].to_numpy()
p95_load = hourly_profiles["p95_load_kw"].to_numpy()
max_load = hourly_profiles["max_load_kw"].to_numpy()
avg_solar = hourly_profiles["avg_solar_kw"].to_numpy()


def plot_diurnal(ax):
    for (start, end), key in zip([(0, 8), (8, 17), (17, 22), (22, 24)], ["night", "day", "evening", "night"]):
        ax.axvspan(start - 0.5, end - 0.5, color=COLORS[key], alpha=0.08)
    ax.plot(hours, max_load, color="firebrick", linestyle="--", label="Historical Max Load (kW)", alpha=0.8)
    ax.plot(hours, p95_load, color="darkorange", label="95th Percentile Load (kW)", linewidth=2)
    ax.plot(hours, avg_load, color="navy", label="Average Factory Load (kW)", linewidth=2.5)
    ax.plot(hours, avg_solar, color="gold", label="Average Rooftop Solar PV (kW)", linewidth=2)

    # Sanctioned ceiling
    ax.axhline(SANCTIONED_KW, color="crimson", linewidth=2.5, linestyle="-", label="LESCO 500 kW Sanctioned Limit")
    ax.fill_between(hours, SANCTIONED_KW, np.maximum(SANCTIONED_KW, max_load), color="crimson", alpha=0.2, label="MDI Penalty Region (>500 kW)")

    ax.set_title("SolarSense EMS — 24-Hour Diurnal Factory Demand & 500 kW LESCO Ceiling", fontsize=14, fontweight="bold")
    ax.set_xlabel("Hour of Day (Asia/Karachi PKT)", fontsize=12)
    ax.set_ylabel("Power (kW)", fontsize=12)
    ax.set_xticks(range(0, 24))
    ax.grid(True, linestyle=":", alpha=0.6)
    ax.legend(loc="upper left", fontsize=8)


fig = plt.figure(figsize=(14, 13))
grid = fig.add_gridspec(3, 2)
plot_diurnal(fig.add_subplot(grid[0, :]))
plot_spike_timeseries(fig.add_subplot(grid[1, :]))
plot_monthly_penalty(fig.add_subplot(grid[2, 0]))
plot_spikes_by_hour(fig.add_subplot(grid[2, 1]))
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "docs" / "samples" / "load_and_mdi_spikes.png", dpi=150, bbox_inches="tight")
print("Plot saved to docs/samples/load_and_mdi_spikes.png")
plt.close()

## 6. Conclusions & Implications for MILP Battery Optimizer
1. **Day Shift Vulnerability:** All 169 MDI breaches fall inside the Day Shift (08:00–17:00), when production machinery runs on top of the base and cooling load. They cluster in the hot months (129 of 169 in Jul–Aug, the rest in May, Jun and Sep), and none fall on Fridays or Sundays. The LESCO seasonal peak window (17:00–21:00 winter, 19:00–23:00 summer, 18:00–22:00 spring/fall; `TariffPolicy.get_peak_hours_for_month`) overlaps the Evening Shift. So the Evening Shift carries the 60 PKR/kWh energy exposure, while the Day Shift carries the MDI exposure.
2. **Baseline Reconciliation:** The 2,360,353.20 PKR baseline comes entirely from grid-connected breaches, with zero charged during blackouts. Under the monthly ratchet LESCO actually bills, the same breaches cost about 239K PKR.
3. **Battery Sizing:** Holding grid import at 495 kW takes about **106 kWh on the worst day** (p95 about 62 kWh) and about **43 kW** of discharge power. The installed 5.12 kWh battery is far too small for this.
4. **P90 Safety Margin:** The battery optimizer must dispatch battery discharge based on the $p90$ load forecast during the 08:00–17:00 day shift to prevent any draw $>500\text{ kW}$.
5. **Outage Islanding:** When blackout risk is elevated, the battery must remain at $\ge 90\%$ state-of-charge to eliminate the 95 PKR/kWh diesel penalty.
